In [ ]:
!pip install polars

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import os
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train_data = pd.read_csv("/content/drive/MyDrive/data_energy/data/train.csv").iloc[:139773, :]

In [ ]:
train_data

,building_id,meter,timestamp,meter_reading
0,0,0,2016-01-01 00:00:00,0.0000
1,1,0,2016-01-01 00:00:00,0.0000
2,2,0,2016-01-01 00:00:00,0.0000
3,3,0,2016-01-01 00:00:00,0.0000
4,4,0,2016-01-01 00:00:00,0.0000
...,...,...,...,...
139768,1259,2,2016-01-03 12:00:00,0.2743
139769,1259,3,2016-01-03 12:00:00,2832.2000
139770,1260,0,2016-01-03 12:00:00,115.3100
139771,1260,1,2016-01-03 12:00:00,0.0000


In [ ]:
weather_data = pd.read_csv("/content/drive/MyDrive/data_energy/data/weather_train.csv")
weather_data

,site_id,timestamp,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2016-01-01 00:00:00,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
1,0,2016-01-01 01:00:00,24.4,NaN,21.1,-1.0,1020.2,70.0,1.5
2,0,2016-01-01 02:00:00,22.8,2.0,21.1,0.0,1020.2,0.0,0.0
3,0,2016-01-01 03:00:00,21.1,2.0,20.6,0.0,1020.1,0.0,0.0
4,0,2016-01-01 04:00:00,20.0,2.0,20.0,-1.0,1020.0,250.0,2.6
...,...,...,...,...,...,...,...,...,...
139768,15,2016-12-31 19:00:00,3.0,NaN,-8.0,NaN,NaN,180.0,5.7
139769,15,2016-12-31 20:00:00,2.8,2.0,-8.9,NaN,1007.4,180.0,7.7
139770,15,2016-12-31 21:00:00,2.8,NaN,-7.2,NaN,1007.5,180.0,5.1
139771,15,2016-12-31 22:00:00,2.2,NaN,-6.7,NaN,1008.0,170.0,4.6


In [ ]:
train_data = pd.merge(train_data, weather_data, on="timestamp", how="left")

In [ ]:
train_data

,building_id,meter,timestamp,meter_reading,site_id,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,0,2016-01-01 00:00:00,0.0,0,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
1,0,0,2016-01-01 00:00:00,0.0,1,3.8,NaN,2.4,NaN,1020.9,240.0,3.1
2,0,0,2016-01-01 00:00:00,0.0,2,15.6,6.0,-5.6,NaN,1015.3,270.0,3.6
3,0,0,2016-01-01 00:00:00,0.0,3,10.0,8.0,2.2,NaN,1021.1,350.0,4.1
4,0,0,2016-01-01 00:00:00,0.0,7,-1.8,NaN,-3.2,NaN,1016.0,280.0,1.5
...,...,...,...,...,...,...,...,...,...,...,...,...
2162819,1260,3,2016-01-03 12:00:00,350.0,11,-1.0,NaN,-1.3,11.0,1001.9,220.0,2.6
2162820,1260,3,2016-01-03 12:00:00,350.0,12,9.4,2.0,5.5,NaN,979.7,210.0,4.0
2162821,1260,3,2016-01-03 12:00:00,350.0,13,-2.8,8.0,-7.2,0.0,1023.9,340.0,5.1
2162822,1260,3,2016-01-03 12:00:00,350.0,14,-2.2,0.0,-5.0,-1.0,1011.1,190.0,1.5


In [ ]:
train_data = train_data.bfill().ffill()

In [ ]:
train_data.describe()

,building_id,meter,meter_reading,site_id,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
count,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06,2.162824e+06
mean,8.124626e+02,6.554408e-01,1.065263e+03,7.289656e+00,5.250760e+00,1.987519e+00,9.289822e-02,5.666249e-02,1.017168e+03,1.862733e+02,3.861168e+00
std,4.257287e+02,9.327310e-01,4.092576e+04,4.505400e+00,8.548267e+00,2.868236e+00,8.860354e+00,9.948011e-01,9.046923e+00,1.099846e+02,2.590887e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.670000e+01,0.000000e+00,-2.000000e+01,-1.000000e+00,9.797000e+02,0.000000e+00,0.000000e+00
25%,4.140000e+02,0.000000e+00,1.000000e+01,3.000000e+00,-1.000000e+00,0.000000e+00,-5.600000e+00,0.000000e+00,1.014400e+03,1.000000e+02,2.100000e+00
50%,9.060000e+02,0.000000e+00,5.275000e+01,7.000000e+00,5.600000e+00,0.000000e+00,-1.700000e+00,0.000000e+00,1.018300e+03,2.200000e+02,3.600000e+00
75%,1.193000e+03,1.000000e+00,1.882570e+02,1.100000e+01,9.400000e+00,4.000000e+00,6.100000e+00,0.000000e+00,1.020700e+03,2.700000e+02,5.100000e+00
max,1.448000e+03,3.000000e+00,3.577180e+06,1.500000e+01,2.830000e+01,8.000000e+00,2.110000e+01,1.300000e+01,1.037800e+03,3.600000e+02,1.400000e+01


In [ ]:
train_data = pl.from_pandas(train_data)

In [ ]:
test_data = pd.read_csv("/content/drive/MyDrive/data_energy/data/test.csv")
weather_data = pd.read_csv("/content/drive/MyDrive/data_energy/data/weather_test.csv")


weather_data

,site_id,timestamp,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2017-01-01 00:00:00,17.8,4.0,11.7,NaN,1021.4,100.0,3.6
1,0,2017-01-01 01:00:00,17.8,2.0,12.8,0.0,1022.0,130.0,3.1
2,0,2017-01-01 02:00:00,16.1,0.0,12.8,0.0,1021.9,140.0,3.1
3,0,2017-01-01 03:00:00,17.2,0.0,13.3,0.0,1022.2,140.0,3.1
4,0,2017-01-01 04:00:00,16.7,2.0,13.3,0.0,1022.3,130.0,2.6
...,...,...,...,...,...,...,...,...,...
277238,15,2018-12-31 19:00:00,3.3,NaN,1.7,NaN,1018.3,150.0,7.7
277239,15,2018-12-31 20:00:00,2.8,NaN,1.1,NaN,1017.8,140.0,5.1
277240,15,2018-12-31 21:00:00,2.8,NaN,1.7,-1.0,1017.2,140.0,6.2
277241,15,2018-12-31 22:00:00,2.8,NaN,2.2,8.0,1016.1,140.0,5.1


In [ ]:
test_data = test_data.iloc[:277243, :]
test_data = pd.merge(test_data, weather_data, on="timestamp", how="left")


In [ ]:
test_data = test_data.bfill().ffill()
test_data

,row_id,building_id,meter,timestamp,site_id,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,0,0,2017-01-01 00:00:00,0,17.8,4.0,11.7,5.0,1021.4,100.0,3.6
1,0,0,0,2017-01-01 00:00:00,1,6.7,6.0,5.2,5.0,1024.1,200.0,5.1
2,0,0,0,2017-01-01 00:00:00,2,16.1,6.0,9.4,5.0,1010.6,310.0,2.1
3,0,0,0,2017-01-01 00:00:00,3,8.9,8.0,-5.6,5.0,1015.3,190.0,8.2
4,0,0,0,2017-01-01 00:00:00,7,-9.4,4.0,-10.6,5.0,1004.5,60.0,4.1
...,...,...,...,...,...,...,...,...,...,...,...,...
4388024,277242,16,0,2017-03-31 13:00:00,11,0.7,6.0,-4.7,0.0,1017.5,70.0,6.2
4388025,277242,16,0,2017-03-31 13:00:00,12,14.1,6.0,7.4,0.0,1001.1,200.0,9.0
4388026,277242,16,0,2017-03-31 13:00:00,13,0.6,6.0,-1.7,0.0,1016.1,360.0,4.6
4388027,277242,16,0,2017-03-31 13:00:00,14,3.3,6.0,2.8,5.0,1015.6,90.0,4.1


In [ ]:
train_data.write_csv("train_data_preprocessed.csv")
test_data.to_csv("test_data_preprocessed.csv", index=False)

In [ ]:
import shutil

shutil.copyfile("/content/train_data_preprocessed.csv", "/content/drive/MyDrive/data_energy/train_data_preprocessed.csv")
shutil.copyfile("/content/test_data_preprocessed.csv", "/content/drive/MyDrive/data_energy/test_data_preprocessed.csv")

'/content/drive/MyDrive/data_energy/test_data_preprocessed.csv'